In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [8]:


class SimpleResNetWithConv(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # 1. Load pretrained ResNet-50[cite: 1]
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        
        # 2. Freeze all pretrained weights so they never change[cite: 1]
        for param in resnet.parameters():
            param.requires_grad = False
            
        # 3. Keep ONLY the convolutional layers (stop before avgpool and fc)[cite: 1]

        self.backbone = nn.Sequential(*list(resnet.children())[:-2]) 
        # you can also do from above way
        # self.backbone = nn.Sequential(
        #     resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
        #     resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
        # )
        
        # 4. Your new trainable layers[cite: 2]
        self.my_conv = nn.Conv2d(in_channels=2048, out_channels=512, kernel_size=3, padding=1)
        self.my_relu = nn.ReLU()
        self.my_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.my_classifier = nn.Linear(512, num_classes)
    
    def forward(self, x):
        # Step 1: Run through frozen ResNet (outputs [Batch, 2048, 7, 7])
        x = self.backbone(x)
        
        # Step 2: Run through your new trainable conv layer (outputs [Batch, 512, 7, 7])[cite: 2]
        x = self.my_relu(self.my_conv(x))
        
        # Step 3: Pool down to 1x1, flatten, and classify (outputs [Batch, num_classes])[cite: 1, 2, 5]
        x = self.my_pool(x)
        x = torch.flatten(x, 1)
        return self.my_classifier(x)

### Layers of ResNet50

[conv1, bn1, relu, maxpool, layer1, layer2, layer3, layer4, avgpool, fc]


A standard ResNet-50 is an assembly line with 10 main parts lined up in order:
[conv1, bn1, relu, maxpool, layer1, layer2, layer3, layer4, avgpool, fc].

The Slicing Syntax: base_model.children() lists those 10 parts. In Python, [:-2] means: "take everything from the start, but leave out the last 2 items."

What Gets Chopped:

The second-to-last item (-2) is avgpool (Global Average Pooling).

The last item (-1) is fc (the original 1,000-class classifier).

Why We MUST Skip avgpool: At layer4, ResNet produces a $7 \times 7$ grid of feature scores. If you let it hit avgpool, it collapses that entire $7 \times 7$ grid down to a single $1 \times 1$ dot. You cannot slide a new $3 \times 3$ convolutional filter across a single $1 \times 1$ point. We chop it off so your new convolutional layer still has a $7 \times 7$ grid to scan.


###
What Each Line in custom_head DoesConv2d(2048, 512, kernel_size=3, padding=1): Your brand-new trainable filter. It slides a $3 \times 3$ window across the $7 \times 7$ grid, compressing ResNet's 2,048 feature maps down to 512 maps.BatchNorm2d(512): Centers and scales the numbers around zero so training stays fast and stable.ReLU(): Clamps negative numbers to zero so only active feature matches pass through.AdaptiveAvgPool2d((1, 1)):

Now we pool. 

It calculates the average of each $7 \times 7$ map, leaving you with 512 single numbers.Flatten(): Unrolls those 512 numbers into a flat 1D line.Linear(512, num_classes): 


The final decision maker that turns those 512 clues into raw prediction scores (logits) for your classes.The Clean, Simple Version (No Cryptic Slicing)Instead of using confusing Python list unpacking (*list(...)[:-2]), you can explicitly name the ResNet layers you want and write a clear, readable forward pass:

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Finetuning

In [ ]:


# Resize to 224x224 and normalize to match ImageNet standard[cite: 1]
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# ----------------------------------------------------
# 3. Setup Training Components
# ----------------------------------------------------
model =  SimpleResNetWithConv(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()

# Hand ONLY the added layers to the optimizer (backbone remains untouched)[cite: 1]
trainable_params = [
    *model.my_conv .parameters(),
    *model.my_classifier.parameters()
]
optimizer = optim.Adam(trainable_params, lr=1e-3)

# ----------------------------------------------------
# 4. Training Loop
# ----------------------------------------------------
model.train()  # Practice mode[cite: 4]
epochs = 2     # Keep it short for demonstration

for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()               # 1. Clear old gradients[cite: 1, 2, 5]
        outputs = model(images)             # 2. Forward pass guess[cite: 1, 2, 4]
        loss = criterion(outputs, labels)   # 3. Grade the guess[cite: 1, 2]
        loss.backward()                     # 4. Assign blame[cite: 1, 2, 5]
        optimizer.step()                    # 5. Turn dials on custom layers[cite: 1, 2, 5]

        total_loss += loss.item()
        
        # Stop early after 50 batches for demonstration
        if i >= 50:
            break

    print(f"Epoch {epoch} | Loss: {total_loss / 50:.4f}")

# ----------------------------------------------------
# 5. Save the Weights Locally
# ----------------------------------------------------
torch.save(model.state_dict(), "custom_resnet_cifar10.pth")

print("Model weights successfully saved to custom_resnet_cifar10.pth")

  4%|█▎                              | 7.05M/170M [01:34<35:32, 76.7kB/s]

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image

# ----------------------------------------------------
# 1. Rebuild the Exact Same Architecture Blueprint
# ----------------------------------------------------
class CustomResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        base_resnet = models.resnet50(weights=None)  # Empty shell, no download needed
        self.backbone = nn.Sequential(*list(base_resnet.children())[:-2])
        self.custom_conv = nn.Conv2d(in_channels=2048, out_channels=512, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.relu(self.custom_conv(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# ----------------------------------------------------
# 2. Load the Saved Weights from Disk
# ----------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomResNet(num_classes=10)

# Pour the saved numbers into the empty architecture
model.load_state_dict(torch.load("custom_resnet_cifar10.pth", map_location=device))
model.to(device)[cite: 6]
model.eval()  # Exam mode (disables dropout / freezes batch statistics)[cite: 1, 4]

# ----------------------------------------------------
# 3. Preprocess a Single Test Image
# ----------------------------------------------------
predict_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create a sample synthetic image (or replace with Image.open("sample.jpg"))
raw_image = Image.new("RGB", (256, 256), color=(120, 150, 200))
input_tensor = predict_transforms(raw_image).unsqueeze(0).to(device)  # Shape: [1, 3, 224, 224][cite: 1]

# ----------------------------------------------------
# 4. Predict Without Tracking Gradients
# ----------------------------------------------------
cifar10_classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

with torch.no_grad():  # Turn off gradient engine to save RAM[cite: 1, 2]
    raw_scores = model(input_tensor)
    predicted_idx = raw_scores.argmax(dim=1).item()

print(f"Predicted Class Index : {predicted_idx}")
print(f"Predicted Label       : {cifar10_classes[predicted_idx]}")